# Notebook 03: Dynamic analysis, traces as data

**Class 04_02, demos D1-D4.** Static features can be faked; behaviour has to be *executed*. A sandbox (Cuckoo,
CAPEv2) runs the file in an instrumented machine and logs the API calls in order. This notebook works on **real**
sandbox traces with API names (nothing synthetic):

* **MalBehavD-V1** (Maniriho et al. 2023): 1,285 benign + 1,285 malware traces, one Cuckoo pipeline. Binary task.
* **Mal-API-2019** (Catak and Yazi 2019): 7,107 malware traces, 8 families (VirusTotal consensus), first 200 calls.
* **Oliveira 2019**: 43,876 traces as integer ids (97.5% malware), used only for the scale check at the end.

Plan: what a trace looks like (D1), how to represent it (D2), sequence models as encoders and a temporal
autoencoder (D3), the shuffle test in detail (D4).

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")

import sys
import time
import warnings

import numpy as np
import polars as pl

sys.path[:0] = [".", "..", "../../src", "../../../src"]
import aasplot as ap
import malwlib as mw

warnings.filterwarnings("ignore")
ap.setup("04")
d_all = mw.load_api_traces()
print(d_all.group_by("source", "family").agg(
    pl.len().alias("n"), pl.col("trace").str.count_matches(" ").add(1).median().alias("median_calls")
).sort("source", "family"))  # fmt: skip
mbd = mw.load_api_traces("mbd")
T = mw.traces(mbd)
y = mbd["malware"].to_numpy()
print(f"MalBehavD: {len(T)} traces, vocabulary {len({w for t in T for w in t})} API names")

## D1. What one trace looks like

Two traces of the same length: a benign installer-like program and a malware sample. Each call is coloured by its
**behaviour class** (file, registry, process, memory, network, library, system, sync, crypto, ui) derived from the
API name prefix (`malwlib.api_category`). The strip is the picture of the trace.

In [ ]:
rng = np.random.default_rng(0)
cats = mw.CATEGORIES
cid = {c: i for i, c in enumerate(cats)}


def strip(tokens, n=60):
    row = [cid[mw.api_category(t)] for t in tokens[:n]]
    return row + [len(cats)] * (n - len(row))


pick = [int(rng.choice(np.flatnonzero((y == k) & (np.array([len(t) for t in T]) >= 40)))) for k in (0, 0, 1, 1)]
M = np.array([strip(T[i]) for i in pick])
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

pal = list(plt.cm.tab20.colors[: len(cats)]) + [(1, 1, 1)]
f, ax = ap.fig("full", 1.4)
ax.imshow(M, aspect="auto", cmap=ListedColormap(pal), vmin=0, vmax=len(cats), interpolation="nearest")
ax.set_yticks(range(4), ["benign A", "benign B", "malware A", "malware B"])
ax.set_xlabel("call index (first 60)")
ax.grid(False)
for s_ in ax.spines.values():
    s_.set_visible(False)
from matplotlib.patches import Patch

ax.legend(handles=[Patch(color=pal[i], label=c) for i, c in enumerate(cats)], ncol=6, loc="upper center",
          bbox_to_anchor=(0.5, -0.32), fontsize=6, handlelength=1, columnspacing=0.8)  # fmt: skip
ap.save(f, "d04_trace_strip")
for i in pick[2:]:
    print("malware head:", " ".join(T[i][:12]))

In [ ]:
# behaviour mix per class: share of calls in each behaviour class (first 200 calls)
groups = {"benign": mbd.filter(pl.col("malware") == 0), "malware (MBD)": mbd.filter(pl.col("malware") == 1)}
fam = mw.load_api_traces("mal2019")
for k in sorted(fam["family"].unique().to_list()):
    groups[k] = fam.filter(pl.col("family") == k)
mix = {}
for k, g in groups.items():
    c = np.zeros(len(cats))
    for t in mw.traces(g, 200):
        for w in t:
            c[cid[mw.api_category(w)]] += 1
    mix[k] = c / c.sum()
for k in ("benign", "malware (MBD)", "worms", "spyware"):
    print(f"{k:14s}", {c: round(float(v), 3) for c, v in zip(cats, mix[k]) if v > 0.03})
f, ax = ap.fig("full", 1.4)
left = np.zeros(len(groups))
for j, c in enumerate(cats):
    v = np.array([mix[k][j] for k in groups])
    ax.barh(range(len(groups)), v, left=left, color=pal[j], label=c, height=0.7)
    left += v
ax.set_yticks(range(len(groups)), list(groups))
ax.invert_yaxis()
ax.set_xlabel("share of calls")
ax.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.45, -0.28), fontsize=6, handlelength=1, columnspacing=0.8)
ap.save(f, "d04_category_mix")

## D2. Representations and the usual models

Four representations of the same trace: **bag of calls** (counts, order lost), **n-grams** (counts of call pairs
and triples, local order kept), the **sequence of ids** (full order, for neural nets) and the **behaviour-class
strip** (11 symbols instead of 291). Toy first: the bigrams of one short trace.

In [ ]:
toy = "ntopenkey ntqueryvaluekey ntclose ntopenkey ntqueryvaluekey ntclose ntcreatefile ntwritefile".split()
vocab_toy = sorted(set(toy))
B = np.zeros((len(vocab_toy), len(vocab_toy)), int)
for a, b in zip(toy, toy[1:]):
    B[vocab_toy.index(a), vocab_toy.index(b)] += 1
f, ax = ap.fig("col", 2.0)
ax.imshow(B, cmap="Blues", vmin=0, vmax=2)
ax.set_xticks(range(len(vocab_toy)), vocab_toy, rotation=90, fontsize=6.6)
ax.set_yticks(range(len(vocab_toy)), vocab_toy, fontsize=6.6)
for (i, j), v in np.ndenumerate(B):
    if v:
        ax.text(j, i, v, ha="center", va="center", fontsize=7.5)
ax.set_xlabel("next call")
ax.set_ylabel("call")
ax.grid(False)
ap.save(f, "d04_bigram_toy")
print("toy trace:", len(toy), "calls ->", len(vocab_toy), "distinct calls,", int((B > 0).sum()), "distinct bigrams")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

idx = np.arange(len(T))
itr, ite = train_test_split(idx, test_size=0.3, stratify=y, random_state=0)
ytr, yte = y[itr], y[ite]
ident = lambda t: t  # noqa: E731  (the traces are already token lists)
bag = CountVectorizer(analyzer=ident)
Atr = bag.fit_transform([T[i] for i in itr])
Ate = bag.transform([T[i] for i in ite])
ng = TfidfVectorizer(analyzer=lambda t: t + [a + ">" + b for a, b in zip(t, t[1:])] +
                     [a + ">" + b + ">" + c for a, b, c in zip(t, t[1:], t[2:])], min_df=3, sublinear_tf=True)  # fmt: skip
Ntr = ng.fit_transform([T[i] for i in itr])
Nte = ng.transform([T[i] for i in ite])
print("features: bag", Atr.shape[1], " n-grams 1-3", Ntr.shape[1])

res = {}
P = {}
for name, (A_, B_, clf) in {
    "bag + LR": (Atr, Ate, LogisticRegression(max_iter=3000)),
    "bag + RF": (Atr, Ate, RandomForestClassifier(300, n_jobs=-1, random_state=0)),
    "n-gram + LR": (Ntr, Nte, LogisticRegression(max_iter=3000)),
    "bag + GBM": (Atr.toarray(), Ate.toarray(), HistGradientBoostingClassifier(max_iter=200, random_state=0)),
}.items():
    t0 = time.time()
    clf.fit(A_, ytr)
    p = clf.predict_proba(B_)[:, 1]
    P[name] = p
    res[name] = (roc_auc_score(yte, p), f1_score(yte, p > 0.5), time.time() - t0)
    print(f"{name:14s} AUC {res[name][0]:.4f}  F1 {res[name][1]:.4f}  fit+predict {res[name][2]:.1f} s")

### Sequence models on ids

The same split for the neural models: a **1-D convolution** (local patterns, order inside a window) and a **GRU**
(reads the whole trace). Both start from an *embedding*: each of the 291 call names becomes a vector that the
network learns. Padding id 0 is masked.

In [ ]:
import keras
from keras import layers

tok = {w: i + 1 for i, w in enumerate(sorted({w for t in T for w in t}))}
V = len(tok) + 1
L = 100  # window: the first 100 calls (what a sandbox with a call budget records)


def ids(seqs, n=L):
    X = np.zeros((len(seqs), n), np.int32)
    for i, s_ in enumerate(seqs):
        s_ = [tok.get(w, 0) for w in s_[:n]]
        X[i, : len(s_)] = s_
    return X


Xtr, Xte = ids([T[i] for i in itr]), ids([T[i] for i in ite])


def make_gru(units=64):
    return keras.Sequential([layers.Input((L,)), layers.Embedding(V, 32, mask_zero=True), layers.GRU(units),
                             layers.Dense(1, activation="sigmoid")])  # fmt: skip


def make_cnn():
    return keras.Sequential([layers.Input((L,)), layers.Embedding(V, 32), layers.Conv1D(64, 5, activation="relu"),
                             layers.GlobalMaxPooling1D(), layers.Dense(1, activation="sigmoid")])  # fmt: skip


def fit_net(make, X, yy, epochs=12, seed=0):
    keras.utils.set_random_seed(seed)
    m_ = make()
    m_.compile("adam", "binary_crossentropy")
    h_ = m_.fit(X, yy, epochs=epochs, batch_size=64, validation_split=0.15, verbose=0)
    return m_, h_


nets = {}
for name, make in {"1-D CNN": make_cnn, "GRU": make_gru}.items():
    t0 = time.time()
    nets[name], h = fit_net(make, Xtr, ytr)
    p = nets[name].predict(Xte, verbose=0)[:, 0]
    P[name] = p
    res[name] = (roc_auc_score(yte, p), f1_score(yte, p > 0.5), time.time() - t0)
    print(f"{name:14s} AUC {res[name][0]:.4f}  F1 {res[name][1]:.4f}  fit+predict {res[name][2]:.1f} s")
    if name == "GRU":
        hist = h.history

In [ ]:
f, ax = ap.fig("col", 1.8)
ep = np.arange(1, len(hist["loss"]) + 1)
ax.plot(ep, hist["loss"], color=ap.BLUE, label="train")
ax.plot(ep, hist["val_loss"], color=ap.RED, label="validation")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()
ap.save(f, "d04_gru_loss")

# bootstrap 95% interval of the AUC on the test traces: models closer than the interval cannot be ranked
rng = np.random.default_rng(1)
ci = {}
for name, p in P.items():
    s_ = [roc_auc_score(yte[b], p[b]) for b in (rng.integers(0, len(yte), len(yte)) for _ in range(200))]
    ci[name] = np.percentile(s_, [2.5, 97.5])
f, ax = ap.fig("full", 1.5)
names = list(P)
a_ = np.array([res[n][0] for n in names])
ax.errorbar(a_, range(len(names)), xerr=[a_ - [ci[n][0] for n in names], [ci[n][1] for n in names] - a_],
            fmt="o", color=ap.BLUE, capsize=2, ms=4)  # fmt: skip
ax.set_yticks(range(len(names)), names)
ax.invert_yaxis()
ax.set_xlabel("AUC with 95% bootstrap interval (771 test traces)")
ap.save(f, "d04_models_auc")
for n in names:
    print(f"{n:14s} AUC {res[n][0]:.4f}  [{ci[n][0]:.4f}, {ci[n][1]:.4f}]")

## D3. A sequence model is an encoder; a temporal autoencoder learns without labels

The GRU is a *layer*. The concept is a **sequence encoder**: it reads a trace of any length and returns one fixed
vector (the last hidden state). Everything after it (a dense layer, logistic regression, a forest) is an ordinary
model on a feature vector. Below: (1) the encoder of the trained GRU feeds a plain logistic regression;
(2) a **temporal autoencoder** is trained on benign traces only and reconstructs the next call; a trace it
cannot reconstruct is anomalous.

In [ ]:
enc = keras.Sequential(nets["GRU"].layers[:-1])  # embedding + GRU, no classifier head
Ztr, Zte = enc.predict(Xtr, verbose=0), enc.predict(Xte, verbose=0)
lr_z = LogisticRegression(max_iter=3000).fit(Ztr, ytr)
pz = lr_z.predict_proba(Zte)[:, 1]
print(f"GRU encoder (64-d) + logistic regression: AUC {roc_auc_score(yte, pz):.4f} F1 {f1_score(yte, pz > .5):.4f}"
      f"   (GRU head: AUC {res['GRU'][0]:.4f})")  # fmt: skip

from sklearn.decomposition import PCA

zz = PCA(2, random_state=0).fit_transform(Zte)
f, ax = ap.fig("col", 2.0)
for k, c, lab in ((0, ap.BLUE, "benign"), (1, ap.RED, "malware")):
    ax.scatter(zz[yte == k, 0], zz[yte == k, 1], s=4, color=c, alpha=0.5, label=lab, lw=0)
ax.set_xlabel("PC1 of the encoder state")
ax.set_ylabel("PC2")
ax.legend(markerscale=2)
ap.save(f, "d04_embed_pca")

In [ ]:
# temporal autoencoder: embedding -> GRU encoder (z, 16-d) -> repeated z -> GRU decoder -> softmax over the call names
ben_tr = Xtr[ytr == 0]


def make_tae(z=16):
    i_ = layers.Input((L,))
    e_ = layers.Embedding(V, 32)(i_)
    h_ = layers.GRU(z)(e_)
    r_ = layers.RepeatVector(L)(h_)
    d_ = layers.GRU(48, return_sequences=True)(r_)
    o_ = layers.Dense(V, activation="softmax")(d_)
    return keras.Model(i_, o_)


keras.utils.set_random_seed(0)
tae = make_tae()
tae.compile("adam", "sparse_categorical_crossentropy")
t0 = time.time()
th = tae.fit(ben_tr, ben_tr, epochs=15, batch_size=64, verbose=0)
print(f"temporal autoencoder trained on {len(ben_tr)} benign traces in {time.time() - t0:.0f} s")


def recon_error(X):
    """Mean cross-entropy of the reconstructed call over the non-padding positions of each trace."""
    pr = tae.predict(X, verbose=0)
    ce = -np.log(np.take_along_axis(pr, X[..., None], 2)[..., 0] + 1e-9)
    m_ = X > 0
    return (ce * m_).sum(1) / m_.sum(1)


e_te = recon_error(Xte)
print(f"reconstruction error: benign {e_te[yte == 0].mean():.2f}, malware {e_te[yte == 1].mean():.2f};"
      f" AUC (no malware label used) {roc_auc_score(yte, e_te):.4f}")  # fmt: skip
f, ax = ap.fig("col", 1.8)
bins = np.linspace(e_te.min(), np.percentile(e_te, 99), 30)
ax.hist(e_te[yte == 0], bins, color=ap.BLUE, alpha=0.7, label="benign")
ax.hist(e_te[yte == 1], bins, color=ap.RED, alpha=0.6, label="malware")
ax.set_xlabel("reconstruction error")
ax.set_ylabel("traces")
ax.legend()
ap.save(f, "d04_ae_scores")

## D4. The shuffle test in detail

Question: *does the model use the order of the calls?* Hypothesis H0: the model only uses **which** calls occur.
If H0 holds, permuting the calls of a test trace must not change the score. Protocol (one seed = one permutation):

1. Window: the first 100 calls of every trace, the same for all models.
2. **Test-only shuffle**: train on the original order, score shuffled test traces. A drop means the model learned
   order. The bag-of-calls model is the control: it ignores order by construction, so its AUC must not move.
3. **Block shuffle**: cut the trace in blocks of $w$ calls and shuffle the blocks. $w=1$ destroys all order,
   $w=100$ leaves the trace intact; the curve shows *how long* the patterns are that the model relies on.
4. **Train and test shuffle**: retrain on shuffled traces. If this model matches the original, no order
   information was needed at all (the task is solved by call sets).
5. Repeat with 5 seeds and report mean and spread; a single shuffle can be lucky.

In [ ]:
T100 = [t[:L] for t in T]
tr_tokens = [T100[i] for i in itr]
te_tokens = [T100[i] for i in ite]
bag2 = CountVectorizer(analyzer=ident)
ng_an = lambda t: (
    t + [a + ">" + b for a, b in zip(t, t[1:])] + [a + ">" + b + ">" + c for a, b, c in zip(t, t[1:], t[2:])]
)  # noqa: E731
ng2 = TfidfVectorizer(analyzer=ng_an, min_df=3, sublinear_tf=True)
lr_bag = LogisticRegression(max_iter=3000).fit(bag2.fit_transform(tr_tokens), ytr)
lr_ng = LogisticRegression(max_iter=3000).fit(ng2.fit_transform(tr_tokens), ytr)
score = {
    "bag + LR": lambda toks: lr_bag.predict_proba(bag2.transform(toks))[:, 1],
    "n-gram + LR": lambda toks: lr_ng.predict_proba(ng2.transform(toks))[:, 1],
    "1-D CNN": lambda toks: nets["1-D CNN"].predict(ids(toks), verbose=0)[:, 0],
    "GRU": lambda toks: nets["GRU"].predict(ids(toks), verbose=0)[:, 0],
}
base = {k: roc_auc_score(yte, fn(te_tokens)) for k, fn in score.items()}
print("original order AUC:", {k: round(v, 4) for k, v in base.items()})

widths = [1, 2, 4, 8, 16, 32, 100]
curve = {k: np.zeros((5, len(widths))) for k in score}
for s_ in range(5):
    rng = np.random.default_rng(100 + s_)
    for j, w in enumerate(widths):
        sh = [mw.block_shuffle(t, w, rng) for t in te_tokens]
        for k, fn in score.items():
            curve[k][s_, j] = roc_auc_score(yte, fn(sh))
for k in score:
    print(f"{k:12s} w=1: {curve[k][:, 0].mean():.4f} +- {curve[k][:, 0].std():.4f}   w=8: {curve[k][:, 3].mean():.4f}   w=100: {curve[k][:, -1].mean():.4f}")  # fmt: skip

f, ax = ap.fig("full", 1.55)
for (k, c), ls in zip(zip(score, [ap.GREY, ap.GOLD, ap.GREEN, ap.RED]), ["--", "-", "-", "-"]):
    m_, s_ = curve[k].mean(0), curve[k].std(0)
    ax.plot(range(len(widths)), m_, ls, color=c, marker="o", ms=3, label=k)
    ax.fill_between(range(len(widths)), m_ - s_, m_ + s_, color=c, alpha=0.2, lw=0)
ax.set_xticks(range(len(widths)), [str(w) if w < 100 else "100 (intact)" for w in widths])
ax.set_xlabel("block width $w$ (calls)")
ax.set_ylabel("test AUC")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.28), fontsize=6.5)
ap.save(f, "d04_shuffle_blocks")

In [ ]:
# shuffle both sides: retrain on shuffled training traces (5 seeds), score shuffled test traces
both = {k: [] for k in ("n-gram + LR", "1-D CNN", "GRU")}
for s_ in range(5):
    rng = np.random.default_rng(200 + s_)
    trs = [mw.block_shuffle(t, 1, rng) for t in tr_tokens]
    tes = [mw.block_shuffle(t, 1, rng) for t in te_tokens]
    m1 = LogisticRegression(max_iter=3000).fit(ng2.transform(trs), ytr)
    both["n-gram + LR"].append(roc_auc_score(yte, m1.predict_proba(ng2.transform(tes))[:, 1]))
    for name, make in (("1-D CNN", make_cnn), ("GRU", make_gru)):
        m_, _ = fit_net(make, ids(trs), ytr, seed=s_)
        both[name].append(roc_auc_score(yte, m_.predict(ids(tes), verbose=0)[:, 0]))
rows = []
for k in both:
    rows.append((k, base[k], curve[k][:, 0].mean(), np.mean(both[k]), np.std(both[k])))
    print(f"{k:12s} original {base[k]:.4f} | test-only shuffle {curve[k][:, 0].mean():.4f} | train+test shuffle {np.mean(both[k]):.4f} +- {np.std(both[k]):.4f}")  # fmt: skip
f, ax = ap.fig("col", 1.9)
x_ = np.arange(len(rows))
ax.bar(x_ - 0.27, [r[1] for r in rows], 0.27, color=ap.BLUE, label="original")
ax.bar(x_, [r[2] for r in rows], 0.27, color=ap.RED, label="test shuffled")
ax.bar(x_ + 0.27, [r[3] for r in rows], 0.27, color=ap.GOLD, label="train+test shuffled", yerr=[r[4] for r in rows], capsize=1.5)  # fmt: skip
ax.set_xticks(x_, [r[0].replace("n-gram + LR", "n-gram\n+ LR") for r in rows])
ax.set_ylim(0.8, 1.0)
ax.set_ylabel("test AUC")
ax.legend(fontsize=6, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.25), columnspacing=0.8, handlelength=1)
ap.save(f, "d04_shuffle_bars")

## D4b. Same test on the family task (Mal-API-2019, 8 classes)

Eight families, first 200 calls, macro-F1 (the mean of the per-class F1). Two questions: does collapsing repeated
calls help (the raw traces are dominated by `ldrgetprocedureaddress` loops), and does order matter for families?

In [ ]:
FT = mw.traces(fam, 200)
fy = fam["family"].to_numpy()
fi_tr, fi_te = train_test_split(np.arange(len(FT)), test_size=0.3, stratify=fy, random_state=0)
rng = np.random.default_rng(7)


def fam_run(prep, shuf=False):
    tr_ = [prep(FT[i]) for i in fi_tr]
    te_ = [prep(FT[i]) for i in fi_te]
    out = {}
    for name, an in (("bag", ident), ("n-gram", ng_an)):
        v = TfidfVectorizer(analyzer=an, min_df=3, sublinear_tf=True, max_features=40000)
        m_ = LogisticRegression(max_iter=2000).fit(v.fit_transform(tr_), fy[fi_tr])
        te_in = [mw.block_shuffle(t, 1, rng) for t in te_] if shuf else te_
        out[name] = f1_score(fy[fi_te], m_.predict(v.transform(te_in)), average="macro")
    return out


fam_res = {}
for label, prep in (("raw", lambda t: t), ("collapsed repeats", mw.collapse_repeats)):
    o, s_ = fam_run(prep), fam_run(prep, shuf=True)
    fam_res[label] = (o, s_)
    print(f"{label:18s} macro-F1 bag {o['bag']:.3f} n-gram {o['n-gram']:.3f} | test shuffled: bag {s_['bag']:.3f} n-gram {s_['n-gram']:.3f}")  # fmt: skip

## Scale check: 43,876 traces as integer ids (Oliveira 2019)

The same shuffle question on a larger corpus (97.5% malware, 307 ids). Bag of calls against bigrams of ids.

In [ ]:
Xs, ys = mw.load_sequences()
Str, Ste, ytr_s, yte_s = train_test_split(Xs, ys, test_size=0.25, stratify=ys, random_state=0)
tokens = lambda X: [[str(v) for v in r if v] for r in X]  # noqa: E731
v_b = TfidfVectorizer(analyzer=ident, sublinear_tf=True)
v_n = TfidfVectorizer(analyzer=lambda t: t + [a + ">" + b for a, b in zip(t, t[1:])], min_df=5, sublinear_tf=True)
tr_t, te_t = tokens(Str), tokens(Ste)
rng = np.random.default_rng(3)
te_sh = [mw.block_shuffle(t, 1, rng) for t in te_t]
scale_res = {}
for name, v in (("bag of ids", v_b), ("bigrams of ids", v_n)):
    m_ = LogisticRegression(max_iter=1000, class_weight="balanced").fit(v.fit_transform(tr_t), ytr_s)
    a0 = roc_auc_score(yte_s, m_.predict_proba(v.transform(te_t))[:, 1])
    a1 = roc_auc_score(yte_s, m_.predict_proba(v.transform(te_sh))[:, 1])
    scale_res[name] = (a0, a1)
    print(f"{name:15s} AUC original {a0:.4f} | test shuffled {a1:.4f} | drop {a0 - a1:.4f}")

In [ ]:
# one picture of the three tasks: relative drop of the n-gram model and of the bag when the test traces are shuffled
f, ax = ap.fig("full", 1.25, ncols=3)
tasks = [
    ("binary (MalBehavD)\nAUC", [(base["bag + LR"], curve["bag + LR"][:, 0].mean()), (base["n-gram + LR"], curve["n-gram + LR"][:, 0].mean())], (0.9, 1.0)),
    ("8 families\nmacro-F1", [(fam_res["raw"][0]["bag"], fam_res["raw"][1]["bag"]), (fam_res["raw"][0]["n-gram"], fam_res["raw"][1]["n-gram"])], (0, 0.7)),
    ("43,876 traces\nAUC", [scale_res["bag of ids"], scale_res["bigrams of ids"]], (0.7, 1.0)),
]  # fmt: skip
for a, (title, vals, yl) in zip(ax, tasks):
    for j, (o_, s2) in enumerate(vals):
        a.bar(j - 0.2, o_, 0.38, color=ap.BLUE, label="original" if j == 0 else None)
        a.bar(j + 0.2, s2, 0.38, color=ap.RED, label="test shuffled" if j == 0 else None)
    a.set_xticks([0, 1], ["bag", "n-gram"])
    a.set_ylim(*yl)
    a.set_title(title, fontsize=7)
ax[0].legend(fontsize=6, loc="upper center", bbox_to_anchor=(1.6, -0.25), ncol=2)
ap.save(f, "d04_shuffle_tasks")

## Takeaways

- Binary detection on MalBehavD: every model sits at AUC .98-.99, intervals overlap, and order is worth less than
  one AUC point (test-only shuffle .989 to .981 for n-grams, GRU unchanged).
- The answer depends on the task: families (8 classes) fall from macro-F1 .55 to .26 when the test traces are
  shuffled, and on the 43,876-trace corpus bigrams drop from AUC .990 to .784 while the bag stays at .953.
- The sequence model is an encoder: its vector works with a plain logistic regression, and a temporal autoencoder
  trained on benign traces alone reaches AUC .76 without a single malware label.
- Run the shuffle test per task and per dataset, with several seeds; never claim "learned order" without it.